# Vision Transformer (ViT) — Easy to Remember

**The whole idea in one line:** Cut an image into patches, turn them into tokens, let attention connect them, then predict a class.

**Memory path:** **Image → Patches → Embeddings → Position + CLS → Attention → Prediction**

This short tutorial explains ViT itself. The final section connects it to the selected [ViTAEv2 paper](https://doi.org/10.1007/s11263-022-01739-w). The small code examples teach the mechanism; they are not results from the paper.

## 1. What is ViT?

A Vision Transformer is a model that **classifies images using a Transformer**. It treats small pieces of the image like a sequence of tokens.

**Remember:** ViT reads *image patches* the way a language Transformer reads *word tokens*.

## 2. Split the image into patches

Divide an image into equal squares. For a **32 × 32** image and **8 × 8** patches, we get **16 patches**: 4 across × 4 down.

**Remember:** A patch is a small piece of the original image.

In [ ]:
import torch
from torch import nn

torch.manual_seed(42)
image = torch.rand(1, 3, 32, 32)  # [batch, RGB channels, height, width]
patches = nn.functional.unfold(image, kernel_size=8, stride=8)
patches = patches.transpose(1, 2)
print("Image:", tuple(image.shape))
print("16 flattened patches:", tuple(patches.shape))  # [1, 16, 192]

## 3. Turn patches into embeddings

A patch is pixels; the Transformer needs vectors. A learned layer converts each patch into an **embedding**.

With RGB images, an 8 × 8 patch has **8 × 8 × 3 = 192** pixel values. We can project those 192 values into a shorter vector, such as **64 numbers**.

**Remember:** Embedding = the patch in the model's vector language.

In [ ]:
projection = nn.Linear(192, 64)
tokens = projection(patches)
print("Patch embeddings:", tuple(tokens.shape))  # [1, 16, 64]

## 4. Add position information

Attention looks at tokens together, but it does not automatically know **where** each patch came from. We add a position vector to each token.

**Remember:** Embedding tells the model *what* the patch contains; position tells it *where* it is.

## 5. Add a CLS token

ViT adds a learnable **CLS token** at the start of the sequence. After attention, this token carries information used for the final image classification.

16 patch tokens + 1 CLS token = **17 tokens**.

**Remember:** CLS is the token used to make the final prediction.

In [ ]:
cls_token = nn.Parameter(torch.zeros(1, 1, 64))
position = nn.Parameter(torch.zeros(1, 17, 64))
sequence = torch.cat([cls_token.expand(tokens.size(0), -1, -1), tokens], dim=1)
sequence = sequence + position
print("CLS + 16 patches:", tuple(sequence.shape))  # [1, 17, 64]

## 6. Let patches communicate with self-attention

Each token makes three vectors: **Query (Q), Key (K), Value (V)**.

A token compares its Q with other tokens' K to decide **which tokens matter**. Then it combines their V vectors. A patch can therefore use information from another part of the image.

**Remember:** Attention asks, “Which other patches should I look at?”

## 7. Use multi-head attention

One attention head learns one pattern of relationships. **Multi-head attention** uses several heads in parallel, allowing the model to learn different relationships.

**Remember:** Multiple heads = multiple ways to connect patches.

## 8. Pass tokens through a Transformer block

A typical block has **attention**, a small **feed-forward network (MLP)**, and **normalization**. Residual connections help information pass through the block.

ViT stacks several such blocks.

**Remember:** The block first mixes information between tokens, then processes each token.

## 9. Make the prediction

Take the final **CLS token** and pass it through a classification layer. For 10 classes, the model returns **10 scores (logits)**.

**Remember:** Final CLS → classification head → class scores.

## 10. See the full path in code

This is a **tiny ViT teaching example** with random weights. It shows the shapes and forward pass. Its prediction has no meaning until the model is trained.

In [ ]:
class TinyViT(nn.Module):
    def __init__(self, image_size=32, patch_size=8, dim=64, num_classes=10):
        super().__init__()
        count = (image_size // patch_size) ** 2
        self.patch_embed = nn.Conv2d(3, dim, patch_size, stride=patch_size)
        self.cls = nn.Parameter(torch.zeros(1, 1, dim))
        self.position = nn.Parameter(torch.randn(1, count + 1, dim) * 0.02)
        block = nn.TransformerEncoderLayer(
            d_model=dim, nhead=4, dim_feedforward=dim * 2,
            batch_first=True, activation="gelu", dropout=0.0
        )
        self.encoder = nn.TransformerEncoder(block, num_layers=2)
        self.head = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2)
        x = torch.cat([self.cls.expand(x.size(0), -1, -1), x], dim=1)
        x = self.encoder(x + self.position)
        return self.head(x[:, 0])

model = TinyViT().eval()
with torch.no_grad():
    scores = model(image)
print("Input:", tuple(image.shape))
print("Class scores:", tuple(scores.shape))  # [1, 10]

## 11. How does ViT learn?

During training, each image comes with a **correct label**. The model predicts scores, calculates a loss, and updates its weights with backpropagation. It learns the patch projection, CLS token, positions, attention, and classifier together.

ViT can be **data hungry**: with limited data, it may benefit from pretraining or architectural ideas that capture image structure.

**Remember:** Label → loss → update weights → better predictions.

## 12. How does the selected paper extend ViT?

The [ViTAEv2 paper](https://doi.org/10.1007/s11263-022-01739-w) starts from a ViT limitation: standard attention does not build in as much knowledge of **nearby image details** and **objects at different scales** as convolution does.

Its solution combines attention with convolution: a **Parallel Convolutional Module (PCM)** adds local details, while a **Pyramid Reduction Module (PRM)** captures multiple scales. ViTAEv2 arranges these ideas in stages.

**Remember:** **ViT = patches + attention. ViTAEv2 = attention + local and multi-scale image cues.**

### Sources

- [ViTAEv2: Vision Transformer Advanced by Exploring Inductive Bias for Image Recognition and Beyond](https://doi.org/10.1007/s11263-022-01739-w) — our selected paper.
- [An Image is Worth 16x16 Words](https://arxiv.org/abs/2010.11929) — the original ViT paper.

The code above is a short educational implementation of ViT, not a reproduction of the selected paper.